# 🎙️ Agentic UIs with CopilotKit — give your deep agent a *face*

In **Track 2** you built a **deep research agent**: it searches arXiv, files papers into a
library that survives across sessions, and turns that library into a podcast series. But you
talked to it through a *notebook* — prompts as Python strings, approvals as hand-edited
`Command(resume=…)` calls.

**This track gives that same agent a real UI.** By the end you'll have a web app where you:

- 💬 chat with the agent in a sidebar,
- 🔎 watch its arXiv searches stream in as **selectable paper cards**,
- ✅ **approve or reject** what gets filed with one click (no more editing `Command(...)`),
- 📚 see your library fill up **live**, and remove papers with an ✕.

All of it built with **[CopilotKit](https://docs.copilotkit.ai)** on the **[AG-UI](https://docs.ag-ui.com)**
protocol. You won't modify the agent at all — the whole workshop is about the *interaction layer*.

> **Runs on free tiers.** You need a free `GOOGLE_API_KEY` ([AI Studio](https://aistudio.google.com/apikey))
> and `TAVILY_API_KEY` ([tavily.com](https://tavily.com)) — the same keys as Track 2. No credit card.

### Agenda (~60 min)

| Part | You'll learn |
|---|---|
| **0** · Setup | one cell: clone, install, keys |
| **1** · The agent, headless | why a notebook loop isn't a product |
| **2** · The AG-UI bridge | expose any agent to any frontend |
| **3** · Connect a React UI | CopilotKit in 3 lines, no runtime |
| **4** · Generative UI | render tool calls as rich cards |
| **5** · Let the human choose | drive the agent *from* the UI |
| **6** · Human-in-the-loop ⭐ | the approve / reject card |
| **7** · Shared state | the living library |
| **8** · Make it yours | theming |
| **9** · Wrap-up | the podcast, and where to go next |

## Part 0 · Setup

Run the cell below **first**. It clones this repo + the Track 2 agent, installs everything
(Python deps, Node.js, and — on first run — the frontend's npm packages), and loads your keys.

> **Tip:** add `GOOGLE_API_KEY` and `TAVILY_API_KEY` to Colab **Secrets** (🔑 in the left
> sidebar, "Notebook access" on) and this cell picks them up automatically. Otherwise it prompts.

In [ ]:
# ▶️ Run this first.
import os, sys, subprocess

REPO = "copilotkit-research-copilot"
AGENT_REPO = "https://github.com/marta-langchain/deep-agents-podcast.git"

if "google.colab" in sys.modules and not os.path.exists("workshop_helpers.py"):
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/sofiasanchez985/{REPO}.git"], check=True)
    os.chdir(REPO)

# The research agent from Track 2 — cloned as-is, never modified. It's the "black box"
# we're giving a face to.
if not os.path.exists("deep-agents-podcast"):
    subprocess.run(["git", "clone", "-q", AGENT_REPO], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "./deep-agents-podcast"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import workshop_helpers as wk
wk.ensure_node()     # installs Node.js if Colab doesn't already have it
wk.load_keys()       # GOOGLE_API_KEY + TAVILY_API_KEY (from Secrets or prompt)
print("\n✅ Setup complete — working dir:", os.getcwd())

## Part 1 · The agent, headless

Here's the agent you built in Track 2 — a `deepagents` graph that searches arXiv and files
papers, **gated by human approval** on every write. Let's look at its shape, then talk to it
the only way we can so far: in Python.

In [ ]:
from workshop_helpers import show_mermaid
from backend.server import build_graph   # our thin wrapper around the Track 2 agent

graph, store = build_graph()
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())   # the compiled LangGraph agent

Notice the **`HumanInTheLoopMiddleware`** node — the agent pauses before it writes anything to
your library. Let's give it a task and watch it stop, waiting for us.

In [ ]:
from uuid import uuid4

cfg = {"configurable": {"thread_id": str(uuid4())}}   # identifies this conversation/run
result = graph.invoke(
    {"messages": [{"role": "user", "content":
        "Find one arXiv paper on epigenetic aging clocks and file it under aging-clocks."}]},
    config=cfg,
)

# The agent paused before writing. Look at what it's asking to do:
interrupt = result.get("__interrupt__")
if interrupt:
    request = interrupt[0].value["action_requests"][0]
    print("⏸  paused before:", request["name"])
    print(request["args"].get("content", request["args"]))
else:
    print("(no pause — the agent didn't try to write this time)")

It's waiting on **your** decision. To **approve** it, run the next cell as-is. To **decline**,
change `"approve"` to `"reject"` first. (You must reuse the same `cfg` — that's how LangGraph
knows which paused run you're answering.)

In [ ]:
from langgraph.types import Command

# ✏️  Change "approve" → "reject" to decline instead.
result = graph.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=cfg,
)
print(result["messages"][-1].text)

That works — but it's a *developer* loop. Prompts are strings, results are `print()`s, and
approving means editing `{"type": "approve"}` and re-running a cell by hand.

> **Key takeaway:** the agent is powerful but *headless*. Everything from here turns that
> pause-and-resume, that search, that library — into a UI a human actually wants to use. (Keep
> this cell in mind: in Part 6 that `Command(resume=…)` becomes a single **Approve** button.)

## Part 2 · The AG-UI bridge

To put a UI on the agent, we need it to *speak to a frontend*. That's what **AG-UI** is: an open
protocol that streams an agent's activity — text, tool calls, state updates, and **interrupts** —
as events any frontend can consume.

In [ ]:
show_mermaid('''
flowchart LR
  A["Deep Agent<br/>(LangGraph graph)"] -- "AG-UI events<br/>text · tools · state · interrupts" --> B["FastAPI endpoint"]
  B -- "HTTP / SSE" --> C["CopilotKit React app"]
  C -- "messages · decisions" --> B
  B --> A
''')

Mounting the agent on an AG-UI endpoint is just a few lines — this is the whole of
`backend/server.py`'s `build_app()`:

```python
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import LangGraphAGUIAgent
from fastapi import FastAPI

# 1. A plain web server to host the agent. It holds no agent logic itself —
#    it's just the door a browser can knock on.
app = FastAPI()

# 2. Wrap the LangGraph graph in an AG-UI adapter. This runs the graph and
#    translates everything it does — text, tool calls, state, interrupts —
#    into standard AG-UI events. The graph is unchanged from Track 2.
agent = LangGraphAGUIAgent(
    name="podcast_agent",
    graph=graph,                          # the exact Track 2 agent from Part 1
    config={"recursion_limit": 100},      # deep agents take many steps (plan → search → subagents)
)

# 3. Mount that adapter as an HTTP/SSE endpoint the React frontend connects to.
add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")
```

Let's start it. It runs as a normal web server in the background.

In [ ]:
import workshop_helpers as wk
wk.start_backend(port=8000)   # the agent, now reachable over AG-UI at :8000

> **Key takeaway:** `LangGraphAGUIAgent` + `add_langgraph_fastapi_endpoint` turn *any* LangGraph
> agent into an AG-UI server without touching the agent's code. That's the seam the whole UI
> plugs into.

## Part 3 · Connect a React UI

CopilotKit has three separable pieces. We use two of them:

- **React SDK** (`@copilotkit/react-core`) — the provider, chat component, and hooks. ⭐ this is the workshop.
- **Python SDK** (`copilotkit`) — what we used in Part 2 to expose the agent.
- **Node runtime** *(optional)* — a server-side proxy. We **skip it**: for a local/workshop app the
  React app connects *directly* to the AG-UI endpoint, which is fewer moving parts.

The connection is the provider wrapping the app (`frontend/src/main.tsx`):

```tsx
import { CopilotKit } from "@copilotkit/react-core/v2";
import { HttpAgent } from "@ag-ui/client";

// Direct connection to the FastAPI endpoint (Vite proxies /agui → :8000).
const podcastAgent = new HttpAgent({ url: "/agui" });

<CopilotKit selfManagedAgents={{ default: podcastAgent }}>
  <App />
</CopilotKit>
```

We'll start with the **simplest possible app — just the chat — and grow it step by step**, writing
the real code into each part and watching the running app change. Here's the starting point
(`frontend/src/App.tsx`): one component, nothing else.

In [ ]:
%%writefile frontend/src/App.tsx
import { CopilotChat } from "@copilotkit/react-core/v2";

export default function App() {
  return (
    <div className="mx-auto flex h-full w-full max-w-3xl flex-col">
      <div className="px-6 py-4">
        <h2 className="font-display text-lg font-semibold text-[#132a2b]">🎙️ Research Copilot</h2>
        <p className="mt-0.5 text-xs text-[#5b7173]">Chat with the deep research agent.</p>
      </div>
      <div className="min-h-0 flex-1">
        <CopilotChat agentId="default" />
      </div>
    </div>
  );
}

Now start the frontend and open the live app 👇 (first run installs npm packages — a couple of
minutes). We'll keep this same running app in view and grow it as we go.

In [ ]:
wk.start_frontend(port=5173)
wk.show_app(port=5173)   # 👈 the running app — right now, just a chat

Try it: type **"Hi — what can you help me research?"** The agent replies right there in the
browser, no Python. It's a plain chat: text in, text out.

> Hold off on *searching for papers* for a moment — that's Part 4, where we add the UI to display
> results. (A search would run fine, but right now the results would have nowhere to render.)
>
> **Key takeaway:** a provider + one `<CopilotChat>` is already a working agent UI. Everything next
> *upgrades* this plain chat into generative UI — and you'll watch each piece appear.

## Part 4 · Generative UI — tool calls become cards

Now give the agent a real task: **searching arXiv**. Its `search_arxiv` tool returns a blob of raw
markdown — not something anyone wants to read in a chat. **`useRenderTool`** intercepts that tool
call and renders it however you like. Write this hook — it maps `search_arxiv` to a list of paper
cards (the `PaperCard` component ships in the repo):

In [ ]:
%%writefile frontend/src/hooks/use-podcast-ui.tsx
import { z } from "zod";
import { useRenderTool } from "@copilotkit/react-core/v2";
import { PaperCard, parsePapers } from "@/components/paper-card";

// Render the agent's arXiv searches as paper cards instead of raw markdown.
export function usePodcastUI() {
  useRenderTool({
    name: "search_arxiv",                        // fires whenever the agent searches
    parameters: z.object({ query: z.string() }),
    render: ({ status, result }) => {
      const papers = status === "complete" ? parsePapers(result) : [];
      return (
        <div className="my-2 space-y-2">
          {papers.map((p, i) => <PaperCard key={p.id || i} paper={p} />)}
        </div>
      );
    },
  });
}

…then call it from `App` so the hook is active:

In [ ]:
%%writefile frontend/src/App.tsx
import { CopilotChat } from "@copilotkit/react-core/v2";
import { usePodcastUI } from "@/hooks/use-podcast-ui";

export default function App() {
  usePodcastUI();   // 👈 registers the generative-UI pieces we build up
  return (
    <div className="mx-auto flex h-full w-full max-w-3xl flex-col">
      <div className="px-6 py-4">
        <h2 className="font-display text-lg font-semibold text-[#132a2b]">🎙️ Research Copilot</h2>
        <p className="mt-0.5 text-xs text-[#5b7173]">Chat with the deep research agent.</p>
      </div>
      <div className="min-h-0 flex-1"><CopilotChat agentId="default" /></div>
    </div>
  );
}

In [ ]:
wk.show_app(port=5173)   # search again — results are cards now

Now search in the app — type **"Find papers on epigenetic aging clocks."** The results stream in
as **paper cards** instead of raw text.

> **Key takeaway:** `useRenderTool("<toolName>")` maps an agent tool call to a React component —
> the agent decides *what* to do, you decide *how it looks*. No agent changes; the tool already
> existed, we just gave its output a face.

## Part 5 · Let the human choose

Plain cards are read-only. Let's make them **selectable** so *you* choose what gets filed. Swap the
render for the `SearchResults` component (it ships in the repo) — it adds checkboxes and a "File
selected" button that drives the agent *from the UI* via **`useAgent`**:

```tsx
// inside SearchResults — the button that files your picks:
const { agent } = useAgent({ agentId: "default" });
agent.addMessage({ id: crypto.randomUUID(), role: "user",
                   content: `Please file these papers:\n${list}` });
agent.runAgent();                 // start a new run with that instruction
```

Update the hook to use it:

In [ ]:
%%writefile frontend/src/hooks/use-podcast-ui.tsx
import { z } from "zod";
import { useRenderTool } from "@copilotkit/react-core/v2";
import { SearchResults } from "@/components/search-results";   // checkboxes + "File selected"

export function usePodcastUI() {
  useRenderTool({
    name: "search_arxiv",
    parameters: z.object({ query: z.string(), max_results: z.number().optional() }),
    render: ({ status, parameters, result }) => (
      <SearchResults
        query={parameters?.query ? String(parameters.query) : undefined}
        result={result}
        status={status}
      />
    ),
  });
}

In [ ]:
wk.show_app(port=5173)   # results now have checkboxes + a "File selected" button

Search again: each result has a checkbox now. Tick a couple and hit **File selected**.

> **Key takeaway:** `useAgent()` gives you the agent handle; `addMessage` + `runAgent` let a button
> (or any UI event) start an agent run. The UI isn't just a chat box — it drives the agent.

## Part 6 · Human-in-the-loop — the approval card ⭐

This is the payoff. The Track 2 agent already **pauses before every write** (`interrupt_on`).
In the notebook that pause was a `Command(resume=…)` you typed. In the browser, CopilotKit's
**`useInterrupt`** turns it into a card.

In [ ]:
show_mermaid('''
sequenceDiagram
  participant U as You
  participant UI as CopilotKit UI
  participant A as Agent (LangGraph)
  A->>A: about to write_file
  A-->>UI: interrupt — run pauses
  UI->>U: Approve / Edit / Reject card
  U->>UI: click Approve
  UI-->>A: resolve → Command(resume=...)
  A->>A: writes the file, continues
''')

Add a second hook — `useInterrupt` — that catches the pause and renders the `ApprovalCard`
(which ships in the repo). `event.value` carries the pending action; `resolve(...)` sends the
decision back and the graph continues — literally the `Command(resume=…)` from Part 1, now a button:

In [ ]:
%%writefile frontend/src/hooks/use-podcast-ui.tsx
import { z } from "zod";
import { useInterrupt, useRenderTool } from "@copilotkit/react-core/v2";
import { SearchResults } from "@/components/search-results";
import { ApprovalCard } from "@/components/approval-card";

export function usePodcastUI() {
  // Search results → selectable paper cards (from Parts 4–5).
  useRenderTool({
    name: "search_arxiv",
    parameters: z.object({ query: z.string(), max_results: z.number().optional() }),
    render: ({ status, parameters, result }) => (
      <SearchResults
        query={parameters?.query ? String(parameters.query) : undefined}
        result={result}
        status={status}
      />
    ),
  });

  // The agent's write-gate pause → an approve / edit / reject card.
  useInterrupt({
    agentId: "default",
    render: ({ event, resolve }) => <ApprovalCard value={event.value} resolve={resolve} />,
    // ApprovalCard's buttons call resolve({ decisions: [{ type: "approve" | "reject" | "edit" }] })
  });
}

In [ ]:
wk.show_app(port=5173)

Now ask it to **find and file a paper**. When the agent tries to write, an **Approve / Edit /
Reject** card appears — the same pause you resumed by hand in Part 1, now one click.

> **Key takeaway:** `useInterrupt` is the render contract for LangGraph interrupts. The agent's
> safety gate — untouched — becomes a one-click approval. This is the single most compelling thing
> CopilotKit does, and Track 2's agent handed it to us for free.

## Part 7 · Shared state — the living library

The left panel is the agent's memory, made visible. The agent files papers into a store; the
backend exposes them at `GET /library`, and the frontend refetches **whenever the agent's state
changes** — so shelves fill in the moment a paper is approved.

```python
# backend/server.py — read the agent's library out of its store
@app.get("/library")
def library():
    return {"shelves": _read_library(store)}
```

```tsx
// frontend/src/components/library-shelf.tsx
const { agent } = useAgent({ agentId: "default" });
useEffect(() => { refresh(); }, [agent?.state]);   // re-read /library on every state update
```

This is also where the app grows from a single chat column into its **two-panel layout**. Rewrite
`App.tsx` to put the live library beside the chat (the `LibraryShelf` component ships in the repo):

In [ ]:
%%writefile frontend/src/App.tsx
import { CopilotChat } from "@copilotkit/react-core/v2";
import { LibraryShelf } from "@/components/library-shelf";
import { usePodcastUI } from "@/hooks/use-podcast-ui";

export default function App() {
  usePodcastUI();
  return (
    <div className="flex h-full w-full">
      {/* Left: the live library — the agent's memory, made visible. */}
      <div className="min-w-0 flex-1">
        <LibraryShelf />
      </div>
      {/* Right: the chat sidebar (frosted glass). */}
      <div className="flex h-full w-[420px] shrink-0 flex-col border-l border-white/60 bg-white/45 backdrop-blur-2xl">
        <div className="px-6 py-4">
          <h2 className="font-display text-lg font-semibold text-[#132a2b]">🎙️ Research Copilot</h2>
          <p className="mt-0.5 text-xs text-[#5b7173]">Find arXiv papers and approve what gets filed into your library.</p>
        </div>
        <div className="min-h-0 flex-1"><CopilotChat agentId="default" /></div>
      </div>
    </div>
  );
}

In [ ]:
wk.show_app(port=5173)

The **library panel appears** on the left. File a paper and watch its shelf fill in a second later.
Click a folder to drill in — each paper is a card with an **✕** to remove it (routed through the
same approval card, which now reads *"remove"*).

> **Key takeaway:** subscribing to `agent.state` keeps the UI in lockstep with the agent. The
> library isn't a copy — it's the agent's own memory, reflected live.

## Part 8 · Make it yours — theming

CopilotKit's chat reads shadcn-style CSS tokens, so matching it to your app is a few variables
(`frontend/src/globals.css`) — no component overrides:

```css
[data-copilotkit] {
  --primary: #0d9488;         /* teal accent — buttons, send, focus */
  --background: transparent;  /* let the app's glass show through   */
  --foreground: #10302f;
  --muted-foreground: #5b7173;
  --radius: 0.9rem;
  font-family: "Inter", system-ui, sans-serif;
}
```

That's how the chat, the cards, and the library all share one look (the "Aurora" glass theme:
a soft mint→lilac gradient, frosted panels, Bricolage Grotesque + Inter).

> **Key takeaway:** theme the whole surface — chat included — from CSS tokens. Consistency is
> a few variables, not a rewrite.

### 🎨 Your turn — edit the theme live

The cell below **is** the app's stylesheet. Change a value, run the cell, and the running app
above updates. Best knobs to play with:

- **`--accent`** — the teal used on buttons, links, badges. Try `#7c3aed` (violet), `#e11d48`
  (rose), or `#ea580c` (orange).
- **the `body { background }` gradient** — swap the colors for a totally different mood.
- **`--glass`** — panel translucency (`0.62` → `0.85` = more solid, less see-through).
- **`--font-display` / `--font-body`** and **`--radius`** (corner roundness).

> The chat retints too, because its tokens (`[data-copilotkit] { … }`) live in this same file.

In [ ]:
%%writefile frontend/src/globals.css
/* 🎨 EDIT ME — change a value below, run this cell, watch the app above update.
   If it doesn't refresh on its own, re-run the show_app cell to reload the iframe. */
@import "tailwindcss";

:root {
  --fg: #132a2b;
  --muted: #5b7173;
  --accent: #0d9488;            /* 👈 the main accent — change me! */
  --accent-strong: #0f766e;
  --accent-soft: rgba(13, 148, 136, 0.12);
  --glass: rgba(255, 255, 255, 0.62);   /* 👈 panel translucency */
  --glass-2: rgba(255, 255, 255, 0.42);
  --glass-border: rgba(255, 255, 255, 0.85);
  --hair: rgba(19, 42, 43, 0.1);
  --radius: 16px;
  --font-display: "Bricolage Grotesque", ui-sans-serif, system-ui, sans-serif;
  --font-body: "Inter", ui-sans-serif, system-ui, -apple-system, sans-serif;
  --font-mono: "JetBrains Mono", ui-monospace, "SF Mono", Menlo, monospace;
}

html, body { height: 100%; }

body {
  margin: 0;
  color: var(--fg);
  font-family: var(--font-body);
  background:                             /* 👈 the mint→lilac gradient — swap the colors! */
    radial-gradient(60% 60% at 12% 6%, rgba(45, 212, 191, 0.28), transparent 60%),
    radial-gradient(55% 55% at 92% 12%, rgba(129, 140, 248, 0.24), transparent 60%),
    radial-gradient(70% 60% at 78% 100%, rgba(94, 234, 212, 0.18), transparent 60%),
    linear-gradient(140deg, #eef5f4 0%, #eef0fb 100%);
  background-attachment: fixed;
}

.font-display { font-family: var(--font-display); font-weight: 600; letter-spacing: -0.01em; }

.glass {
  background: var(--glass);
  backdrop-filter: blur(14px) saturate(1.2);
  -webkit-backdrop-filter: blur(14px) saturate(1.2);
  border: 1px solid var(--glass-border);
}

.glass-card {
  background: var(--glass);
  backdrop-filter: blur(10px) saturate(1.15);
  -webkit-backdrop-filter: blur(10px) saturate(1.15);
  border: 1px solid var(--glass-border);
  box-shadow: 0 10px 30px -18px rgba(15, 60, 60, 0.45);
}

/* The chat inherits your palette — its shadcn tokens live here and this file
   loads after CopilotKit's stylesheet, so these win. */
[data-copilotkit] {
  --background: transparent;
  --foreground: #10302f;
  --card: var(--glass);
  --card-foreground: #10302f;
  --popover: rgba(255, 255, 255, 0.9);
  --popover-foreground: #10302f;
  --primary: #0d9488;                 /* 👈 tip: match this to --accent above */
  --primary-foreground: #ffffff;
  --secondary: rgba(13, 148, 136, 0.1);
  --secondary-foreground: #0f766e;
  --muted: rgba(19, 42, 43, 0.06);
  --muted-foreground: #5b7173;
  --accent: rgba(13, 148, 136, 0.12);
  --accent-foreground: #0f766e;
  --border: rgba(19, 42, 43, 0.12);
  --input: rgba(255, 255, 255, 0.6);
  --ring: #0d9488;
  --radius: 0.9rem;
  font-family: var(--font-body);
}

[data-copilotkit] h1.cpk\:text-center { font-size: 16px; }

Scroll back up to the running app to see your changes. (Vite hot-reloads CSS; if the iframe
doesn't update, just re-run the `show_app` cell.)

> **Key takeaway:** one stylesheet themes the *entire* surface — your components **and** the
> CopilotKit chat — because it all reads the same CSS variables.

### 🧩 Bolder: edit an actual component

Theming is CSS — now edit *real UI*. The cell below is the **`PaperCard`** component that renders
every paper (in search results **and** in library folders). Tweak the JSX, run the cell, and every
card restyles. Things to try:

- add a colored **left border** (`border-l-4 border-teal-500`) for a "rail" look,
- put a 📄 emoji before the title, or bump it to `text-base`,
- show the abstract in full (drop `line-clamp-3`).

> It's real TSX, so a typo will show a Vite error overlay. **Broke it? Just re-run this cell
> unchanged to restore the original.**

In [ ]:
%%writefile frontend/src/components/paper-card.tsx
// 🧩 EDIT ME — this renders every paper card (search results + library folders).
// Leave the Paper type and parsePapers helper below as-is; edit the PaperCard JSX
// at the bottom. Broke it? Re-run this cell unchanged to restore.

export interface Paper {
  id: string;
  title: string;
  authors?: string;
  published?: string;
  abstract?: string;
}

// Parses the markdown that search_arxiv returns into structured papers.
export function parsePapers(result: string | undefined): Paper[] {
  if (!result || typeof result !== "string") return [];
  const blocks = result.split(/^###\s+/m).slice(1);
  return blocks.map((block) => {
    const [titleLine, ...rest] = block.split("\n");
    const field = (name: string) => {
      const line = rest.find((l) => l.trim().toLowerCase().startsWith(`- ${name}:`));
      return line ? line.slice(line.indexOf(":") + 1).trim() : undefined;
    };
    return {
      title: titleLine.trim(),
      id: field("id") ?? "",
      authors: field("authors"),
      published: field("published"),
      abstract: field("abstract"),
    };
  });
}

// ▼▼▼  EDIT FROM HERE DOWN  ▼▼▼
export function PaperCard({
  paper,
  onRemove,
  removing,
}: {
  paper: Paper;
  onRemove?: () => void;
  removing?: boolean;
}) {
  const arxivUrl = paper.id ? `https://arxiv.org/abs/${paper.id}` : undefined;
  return (
    <div className="overflow-hidden rounded-2xl border border-white/70 bg-white/60 shadow-[0_10px_30px_-18px_rgba(15,60,60,0.45)] backdrop-blur-md">
      <div className="space-y-1.5 p-3.5">
        <div className="flex items-start justify-between gap-2">
          <h3 className="font-display text-sm font-semibold leading-snug text-[#132a2b]">{paper.title}</h3>
          <div className="flex shrink-0 items-center gap-1.5">
            {paper.id && (
              <span className="rounded-md bg-teal-600/10 px-1.5 py-0.5 font-mono text-[10px] text-teal-700">
                {paper.id}
              </span>
            )}
            {onRemove && (
              <button
                title="Remove from library"
                disabled={removing}
                onClick={onRemove}
                className="flex h-5 w-5 items-center justify-center rounded-full text-[#8aa0a0] hover:bg-red-50 hover:text-red-600 disabled:opacity-40"
              >
                {removing ? "…" : "✕"}
              </button>
            )}
          </div>
        </div>
        {(paper.authors || paper.published) && (
          <p className="text-xs text-[#6b8382]">
            {paper.authors}
            {paper.authors && paper.published ? " · " : ""}
            {paper.published}
          </p>
        )}
        {paper.abstract && (
          <p className="text-xs leading-relaxed text-[#5b7173] line-clamp-3">{paper.abstract}</p>
        )}
        {arxivUrl && (
          <a href={arxivUrl} target="_blank" rel="noreferrer"
             className="inline-block text-xs font-medium text-teal-700 hover:underline">
            View on arXiv →
          </a>
        )}
      </div>
    </div>
  );
}

Scroll up to the app and run a search — your restyled cards appear in the chat and in every
folder. (Re-run `show_app` if the iframe doesn't refresh.)

> **Key takeaway:** the generative-UI components are just React. Once the agent → AG-UI → CopilotKit
> pipe is in place, *designing the experience* is ordinary frontend work — edit a component, see it
> everywhere it's used.

## Part 9 · Wrap-up & next steps

You took the headless Track 2 agent and gave it a real face — with **zero changes to the agent**:

- **AG-UI** exposed it to any frontend (`LangGraphAGUIAgent`)
- **`<CopilotChat>`** connected a React app directly, no runtime
- **`useRenderTool`** turned searches into selectable paper cards
- **`useAgent`** let the UI drive the agent
- **`useInterrupt`** turned the approval gate into a one-click card ⭐
- **`agent.state`** kept the library live

### Where to go next
- **The podcast 🎧** — the agent's `paper-podcast` skill generates an episode briefing from your
  library. Drop it into **[NotebookLM](https://notebooklm.google.com)** for a two-host audio
  overview (no code — that's the Track 2 finale).
- **Add your own generative UI** — try a `useRenderTool` for `get_arxiv_paper`, or a shared-state
  "reading progress" panel.
- **Docs:** [CopilotKit](https://docs.copilotkit.ai) · [AG-UI](https://docs.ag-ui.com) ·
  [Generative UI](https://docs.copilotkit.ai/concepts/generative-ui-overview) ·
  [Human-in-the-loop](https://docs.copilotkit.ai/human-in-the-loop)

*Same agent as Track 2 — now with a face. 🎙️*